# Лабораторная работа 7
## Деревья

Выполнение заданий по порядку.


## Задача №1: Парсер логических выражений и evaluate
Операторы: `не` (унарный), `и`, `или`. Операнды: `истина`, `ложь`. Скобки поддерживаются.


In [8]:
def tokenize(expr: str):
    expr = expr.replace('(', ' ( ').replace(')', ' ) ')
    return expr.lower().split()

def buildParseTree(expr: str):
    tokens = tokenize(expr)
    i = 0
    
    def parse_expr():
        nonlocal i
        left = parse_term()
        while i < len(tokens) and tokens[i] in ('и', 'или'):
            op = tokens[i]
            i += 1
            right = parse_term()
            left = [op, left, right]
        return left
    
    def parse_term():
        nonlocal i
        if i >= len(tokens):
            return None
        
        # Унарный оператор 'не'
        if tokens[i] == 'не':
            i += 1
            operand = parse_term()
            return ['не', operand]
        
        # Скобки
        if tokens[i] == '(':
            i += 1
            result = parse_expr()
            if i < len(tokens) and tokens[i] == ')':
                i += 1
            return result
        
        # Операнды
        if tokens[i] in ('истина', 'ложь'):
            val = True if tokens[i] == 'истина' else False
            i += 1
            return [val]
        
        return None
    
    return parse_expr()

def evaluate(parse_tree):
    if not parse_tree or len(parse_tree) == 0:
        return None
    op = parse_tree[0]
    if op in (True, False):
        return op
    if op == 'не':
        if len(parse_tree) > 1:
            return not evaluate(parse_tree[1])
        return None
    if op in ('и', 'или'):
        if len(parse_tree) >= 3:
            left = evaluate(parse_tree[1])
            right = evaluate(parse_tree[2])
            if op == 'и':
                return left and right
            if op == 'или':
                return left or right
        return None
    return None

def printexp(parse_tree):
    if not parse_tree or len(parse_tree) == 0:
        return ''
    op = parse_tree[0]
    if op in (True, False):
        return 'истина' if op else 'ложь'
    if op == 'не':
        if len(parse_tree) > 1:
            return f"(не {printexp(parse_tree[1])})"
        return "(не )"
    if op in ('и', 'или'):
        left = printexp(parse_tree[1]) if len(parse_tree) > 1 else ''
        right = printexp(parse_tree[2]) if len(parse_tree) > 2 else ''
        return f"({left} {op} {right})"
    return str(op)

examples = ["(истина и (ложь или не ложь))", "(не (истина или ложь))", "((истина или ложь) и (не ложь))"]
for ex in examples:
    tree = buildParseTree(ex)
    try:
        result = evaluate(tree)
        print(f"{ex} -> {printexp(tree)} = {result}")
    except Exception as e:
        print(f"Ошибка в '{ex}': {e}")
        print(f"Дерево: {tree}")


(истина и (ложь или не ложь)) -> (истина и (ложь или (не ложь))) = True
(не (истина или ложь)) -> (не (истина или ложь)) = False
((истина или ложь) и (не ложь)) -> ((истина или ложь) и (не ложь)) = True


## Задача №2: Двоичное дерево поиска и дубликаты


In [9]:
class BSTNode:
    def __init__(self, key, value=None):
        self.key, self.value = key, value
        self.left = self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def _put(self, node, key, value):
        if node is None:
            return BSTNode(key, value)
        if key == node.key:
            node.value = value  # замена при дубликате
        elif key < node.key:
            node.left = self._put(node.left, key, value)
        else:
            node.right = self._put(node.right, key, value)
        return node

    def put(self, key, value=None):
        self.root = self._put(self.root, key, value)

    def _inorder(self, node, acc):
        if not node:
            return
        self._inorder(node.left, acc)
        acc.append(node.key)
        self._inorder(node.right, acc)

    def keys(self):
        acc = []
        self._inorder(self.root, acc)
        return acc


def has_no_duplicates(tree: BinarySearchTree):
    keys = tree.keys()
    return len(keys) == len(set(keys))

# Проверка: вставка дубликата заменяет значение и не создает копий
bst = BinarySearchTree()
for k, v in [(5,'a'), (3,'b'), (7,'c'), (3,'new_b'), (5,'new_a')]:
    bst.put(k, v)
print('Ключи без дублей:', bst.keys())
print('Нет дубликатов?', has_no_duplicates(bst))


Ключи без дублей: [3, 5, 7]
Нет дубликатов? True


## Задача №3: Игра «Животные»


In [ ]:
from collections import deque

class AnimalNode:
    def __init__(self, text, is_question=False):
        self.text = text
        self.is_question = is_question
        self.yes = None
        self.no = None

def ask_yes_no(prompt, scripted=None):
    if scripted is not None:
        return scripted.popleft()
    return input(prompt + ' (да/нет): ').strip().lower().startswith('д')

def play(root: AnimalNode, scripted_answers=None):
    answers = deque(scripted_answers) if scripted_answers else None
    node = root
    path = []
    while node.is_question:
        ans = ask_yes_no(node.text, answers)
        path.append((node.text, ans))
        node = node.yes if ans else node.no
    # достигли животного
    correct = ask_yes_no(f"Это {node.text}?", answers)
    if correct:
        print('Угадал!')
        return root
    # обучение
    new_animal = 'змея' if answers else input('Какое животное вы загадали? ')
    diff_question = 'У него нет лап?' if answers else input(f"Вопрос, отличающий {new_animal} от {node.text}: ")
    yes_for_new = ask_yes_no(f"Для {new_animal} ответ 'да' на вопрос '{diff_question}'?", answers)
    new_q = AnimalNode(diff_question, is_question=True)
    new_an = AnimalNode(new_animal)
    if yes_for_new:
        new_q.yes, new_q.no = new_an, node
    else:
        new_q.yes, new_q.no = node, new_an
    # вставляем в дерево
    # найдём родителя последнего вопроса, если путь пуст, новый корень
    if not path:
        return new_q
    # восстановление пути
    cur = root
    for q_text, ans in path[:-1]:
        cur = cur.yes if ans else cur.no
    if path:
        last_q, last_ans = path[-1]
        if last_ans:
            cur.yes = new_q
        else:
            cur.no = new_q
    return root

# Стартовое дерево (из примера): вопрос и два ответа
root = AnimalNode('Это млекопитающее?', True)
root.yes = AnimalNode('жираф')
root.no = AnimalNode('крокодил')

# Скрипт ответа для проверки: загадана змея
script = [False, False, True, True]  # нет на млекопитающее, нет это крокодил?, да новое животное змея, да вопрос про лапы
root = play(root, script)
print('Текущее дерево после обучения:')
print(f"Корень: {root.text}, да -> {root.yes.text}, нет -> {root.no.text}")


Текущее дерево после обучения:
Корень: Это млекопитающее?, да -> жираф, нет -> У него нет лап?


## Задача №4: Ограниченная двоичная куча


In [11]:
import heapq

class BoundedMinHeap:
    def __init__(self, limit):
        self.limit = limit
        self.data = []

    def push(self, item):
        if len(self.data) < self.limit:
            heapq.heappush(self.data, item)
        else:
            # если новый элемент важнее (больше), заменим наименьший
            if item > self.data[0]:
                heapq.heapreplace(self.data, item)

    def pop(self):
        return heapq.heappop(self.data) if self.data else None

    def __len__(self):
        return len(self.data)

bh = BoundedMinHeap(3)
for x in [5, 1, 7, 2, 9, 3]:
    bh.push(x)
print('Содержимое ограниченной кучи (3 наибольших):', sorted(bh.data))


Содержимое ограниченной кучи (3 наибольших): [5, 7, 9]


## Задача №5: Max heap


In [12]:
class BinaryHeapMax:
    def __init__(self):
        self.data = []

    def insert(self, item):
        heapq.heappush(self.data, -item)

    def findMax(self):
        return -self.data[0] if self.data else None

    def delMax(self):
        return -heapq.heappop(self.data) if self.data else None

    def size(self):
        return len(self.data)

bh_max = BinaryHeapMax()
for x in [5, 1, 7, 2, 9, 3]:
    bh_max.insert(x)
print('Макс-куча, максимум:', bh_max.findMax())


Макс-куча, максимум: 9


## Задача №6: PriorityQueue на BinaryHeap


In [13]:
class BinaryHeap:
    def __init__(self):
        self.data = []

    def insert(self, item):
        heapq.heappush(self.data, item)

    def delMin(self):
        return heapq.heappop(self.data) if self.data else None

    def size(self):
        return len(self.data)

class PriorityQueue:
    def __init__(self):
        self.heap = BinaryHeap()

    def enqueue(self, priority_item):
        self.heap.insert(priority_item)

    def dequeue(self):
        return self.heap.delMin()

    def __len__(self):
        return self.heap.size()

pq = PriorityQueue()
for item in [5, 1, 7, 2, 9, 3]:
    pq.enqueue(item)
print('Извлечения из очереди с приоритетом:')
while len(pq):
    print(pq.dequeue(), end=' ')
print()


Извлечения из очереди с приоритетом:
1 2 3 5 7 9 
